# 3D Human Reconstruction System

Girish Krishnan, Benny Cai, Bang Du, Kunyao Chen, and Truong Nguyen

## Introduction

This project involves the reconstruction of a 3D human model using color images and corresponding depth maps.

## Dependencies

Run the cell below to install all required libraries

In [1]:
! pip install -r requirements.txt

Importing relevant libraries

In [1]:
# Importing the libraries
import pyrealsense2 as rs
import os
import cv2 as cv
import numpy as np
import json
import shutil
import open3d as o3d
import matplotlib.pyplot as plt
import cv2.aruco as aruco
from itertools import combinations
from scipy.spatial.transform import Rotation as R
import glob
from Camera import Camera
import sys


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Constants

In [2]:
# Constant parameters used in Aruco methods
ARUCO_PARAMETERS = aruco.DetectorParameters_create()
ARUCO_DICT = aruco.Dictionary_get(aruco.DICT_5X5_250)

CHARUCOBOARD_ROWCOUNT = 9
CHARUCOBOARD_COLCOUNT = 12

distCoeffs = np.array([0.0, 0.0, 0.0, 0.0, 0.0])

# Create grid board object we're using in our stream
CHARUCO_BOARD = aruco.CharucoBoard_create(
    squaresX=CHARUCOBOARD_COLCOUNT,
    squaresY=CHARUCOBOARD_ROWCOUNT,
    squareLength=0.060,
    markerLength=0.044,
    dictionary=ARUCO_DICT)

**Optional**: In case one or more cameras are not working as expected, please uncomment and run the following cell to perform a hardware reset for each camera.

In [3]:
ctx = rs.context()
devices = ctx.query_devices()
for dev in devices:
    dev.hardware_reset()

Setting up constants and variables

In [4]:

serial_numbers = []
pipelines = []
configs = []
align = []
profiles = []

# load configuration parameters

with(open("./configuration_parameters.json")) as f:
    configuration_parameters = json.load(f)
    f.close()

NUM_CALIB_IMGS = configuration_parameters["num_calibration_imgs"]
CHECKERBOARD = (
configuration_parameters["checkerboard_dimensions"][0], configuration_parameters["checkerboard_dimensions"][1])
CHECKERBOARD_SIZE = configuration_parameters["checkerboard_size_mm"]  # units: millimeters
CHECKERBOARD_SIZE *= 0.001
IMAGE_TYPE = configuration_parameters["img_file_type"]
THRESHOLD = configuration_parameters["threshold"]

criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

objp = np.zeros((CHECKERBOARD[1] * CHECKERBOARD[0], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[1], 0:CHECKERBOARD[0]].T.reshape(-1, 2)
objp = CHECKERBOARD_SIZE * objp

ctx = rs.context()

Run this cell to browse through all currently connected Intel RealSense devices and reset the configuration parameters and the current image directories.

In [5]:
# get serial numbers of connected cameras
if len(ctx.devices) > 0:

    for device_num in range(len(ctx.devices)):
        
        print ('Found device: ', ctx.devices[device_num].get_info(rs.camera_info.name), ' ', ctx.devices[device_num].get_info(rs.camera_info.serial_number))
        serial_numbers.append(ctx.devices[device_num].get_info(rs.camera_info.serial_number))
        pipelines.append(rs.pipeline())
        configs.append(rs.config())
        configs[device_num].enable_device(ctx.devices[device_num].get_info(rs.camera_info.serial_number))
        configs[device_num].enable_stream(rs.stream.depth, 640,480, rs.format.z16, 60)
        configs[device_num].enable_stream(rs.stream.color, 640,480, rs.format.bgr8, 60)
        configs[device_num].enable_stream(rs.stream.infrared, 640, 480, rs.format.y8, 60)
        #os.chmod(ctx.devices[device_num].get_info(rs.camera_info.serial_number), 0o777)
        if os.path.exists(ctx.devices[device_num].get_info(rs.camera_info.serial_number)):
            shutil.rmtree(ctx.devices[device_num].get_info(rs.camera_info.serial_number))
        if not os.path.exists(ctx.devices[device_num].get_info(rs.camera_info.serial_number)):
            os.makedirs(ctx.devices[device_num].get_info(rs.camera_info.serial_number))
        
        if not os.path.exists(ctx.devices[device_num].get_info(rs.camera_info.serial_number) + "/sample_images"):
            os.makedirs(ctx.devices[device_num].get_info(rs.camera_info.serial_number) + "/sample_images")
        if not os.path.exists(ctx.devices[device_num].get_info(rs.camera_info.serial_number) + "/calibration_images"):
            os.makedirs(ctx.devices[device_num].get_info(rs.camera_info.serial_number) + "/calibration_images")

        configs[device_num].enable_record_to_file('./' + ctx.devices[device_num].get_info(rs.camera_info.serial_number) + '/video.bag')

        # Align objects
        align_to = rs.stream.depth  # align to depth frame
        align.append(rs.align(align_to))
        pipelines[device_num].start(configs[device_num])

        # enable IR emitter and auto exposure
        profile = pipelines[device_num].get_active_profile()

        profiles.append(profile)
        color_profile = rs.video_stream_profile(profile.get_stream(rs.stream.color))
        color_intrinsics = color_profile.get_intrinsics()
        # depth_profile = rs.video_stream_profile(profile.get_stream(rs.stream.depth))
        # depth_intrinsics = depth_profile.get_intrinsics()
        ir_profile = rs.video_stream_profile(profile.get_stream(rs.stream.infrared))
        ir_intrinsics = ir_profile.get_intrinsics()

        s_num = ctx.devices[device_num].get_info(rs.camera_info.serial_number)
        configuration_parameters["cams"][s_num] = {}
        configuration_parameters["cams"][s_num]["intrinsics"] = {}
        configuration_parameters["cams"][s_num]["intrinsics"]["img_size"] = [640, 480]
        configuration_parameters["cams"][s_num]["intrinsics"]["focal_length"] = [color_intrinsics.fx,
                                                                                       color_intrinsics.fy]
        configuration_parameters["cams"][s_num]["intrinsics"]["img_center"] = [color_intrinsics.ppx,
                                                                                     color_intrinsics.ppy]
        # configuration_parameters["cams"][s_num]["intrinsics"]["depth_focal_length"] = [depth_intrinsics.fx,
        #                                                                                depth_intrinsics.fy]
        # configuration_parameters["cams"][s_num]["intrinsics"]["depth_img_center"] = [depth_intrinsics.ppx,
        #                                                                              depth_intrinsics.ppy]
        configuration_parameters["cams"][s_num]["intrinsics"]["ir_focal_length"] = [ir_intrinsics.fx,
                                                                                       ir_intrinsics.fy]
        configuration_parameters["cams"][s_num]["intrinsics"]["ir_img_center"] = [ir_intrinsics.ppx,
                                                                                     ir_intrinsics.ppy]

        device = profile.get_device()
        depth_sensor = device.query_sensors()[0]
        emitter = depth_sensor.get_option(rs.option.emitter_enabled)
        print("old emitter = ", emitter)
        depth_sensor.set_option(rs.option.emitter_enabled, 1)  # enable IR emitter
        emitter1 = depth_sensor.get_option(rs.option.emitter_enabled)
        print("new emitter = ", emitter1)
        depth_sensor.set_option(rs.option.enable_auto_exposure, True)  # enable auto exposure

        depth_sensor.set_option(rs.option.laser_power, 360)  # max laser power
        print("laser power: ", depth_sensor.get_option(rs.option.laser_power))

    json.dump(configuration_parameters, open("configuration_parameters.json", "w"), indent = 4)    

else:

    print("No Intel Device connected")
    exit(-1)


Found device:  Intel RealSense D415   839112060979
old emitter =  1.0
new emitter =  1.0
laser power:  360.0
Found device:  Intel RealSense D415   839212060064
old emitter =  1.0
new emitter =  1.0
laser power:  360.0
Found device:  Intel RealSense D415   828612060381
old emitter =  1.0
new emitter =  1.0
laser power:  360.0
Found device:  Intel RealSense D415   839112061696
old emitter =  1.0
new emitter =  1.0
laser power:  360.0


Capturing: obtains color and depth frames from each camera and displays frames in a new pop-up window.

Check that the human model is still and visible by all cameras.

Press the spacebar to capture images and corresponding depth maps from all connected cameras.

In [6]:

"""
START RECORDING SOME FRAMES

"""
raw_color_images = len(serial_numbers) * [0]
color_images = len(serial_numbers) * [0]
color_frames = len(serial_numbers) * [0]
depth_images = len(serial_numbers) * [0]
depth_frames = len(serial_numbers) * [0]
depth_colormaps = len(serial_numbers) * [0]

try:
    while True:

        for i in range(len(serial_numbers)):

            frames = pipelines[i].wait_for_frames()
            raw_color_frame = frames.get_color_frame()
            aligned_frames = align[i].process(frames)
            color_frames[i] = aligned_frames.get_color_frame()
            depth_frames[i] = aligned_frames.get_depth_frame()
            timestamp = depth_frames[i].get_timestamp()
            if not color_frames[i] or not depth_frames[i]:
                continue
        
        for i in range(len(serial_numbers)):
            depth_frames[i] = rs.decimation_filter(1).process(depth_frames[i])
            depth_frames[i] = rs.disparity_transform(True).process(depth_frames[i])
            depth_frames[i] = rs.spatial_filter().process(depth_frames[i])
            depth_frames[i] = rs.temporal_filter().process(depth_frames[i])
            depth_frames[i] = rs.disparity_transform(False).process(depth_frames[i])

            #print("Camera " + str(i) + ": " + str(timestamp))

            # Convert images to numpy arrays
            raw_color_images[i] = np.asanyarray(raw_color_frame.get_data())
            color_images[i] = np.asanyarray(color_frames[i].get_data())
            depth_images[i] = np.asanyarray(depth_frames[i].get_data())

            depth_colormaps[i] = cv.applyColorMap(cv.convertScaleAbs(depth_images[i], alpha=0.03), cv.COLORMAP_JET)

        # Stack all images horizontally
        stacked_color_images = np.hstack(tuple(color_images))
        stacked_depth_images = np.hstack(tuple(depth_colormaps))
        images = np.vstack((stacked_color_images, stacked_depth_images))
        cv.namedWindow('RealSense', cv.WINDOW_NORMAL)
        cv.imshow('RealSense', images)

        ch = cv.waitKey(1)
        if ch==32:

            for i in range(len(serial_numbers)):
                cv.imwrite('./' + serial_numbers[i] + '/sample_images/raw_image.jpg', raw_color_images[i])
                cv.imwrite('./' + serial_numbers[i] + '/sample_images/image.jpg', color_images[i])
                np.save('./' + serial_numbers[i] + '/sample_images/depth_map.npy', depth_images[i])
                cv.imwrite('./' + serial_numbers[i] + '/sample_images/depth.png', depth_colormaps[i])
            break


finally:

    # Stop streaming
    for pipeline in pipelines:
        pipeline.stop()

    cv.destroyAllWindows()

    

## Calibration Images (ChArUco)

Run the following cell to begin capturing calibration images. The variable NUM_CALIB_IMGS indicates the total number of calibration images captured. Each image must contain the ChArUco board visible from at least two cameras. Press the spacebar to capture an image.

In [8]:
NUM_CAMS = len(configuration_parameters["cams"])

if NUM_CAMS == 0:
    print("No camera images found. Please check the directory.")
    exit(-1)

# Extracting path of individual image stored in a given directory
images = []
for i in range(NUM_CAMS):
    images.append(glob.glob("./" + serial_numbers[i] + "/calibration_images" + "/*" + IMAGE_TYPE))

image_pairs = combinations(range(NUM_CAMS), 2)  # finding all distinct pairs of cameras

for pair in image_pairs:
    x = pair[0]
    y = pair[1]

    cam1_f = configuration_parameters["cams"][serial_numbers[x]]["intrinsics"]["ir_focal_length"]
    cam1_c = configuration_parameters["cams"][serial_numbers[x]]["intrinsics"]["ir_img_center"]
    cam1_mtx = np.array([
        [cam1_f[0], 0, cam1_c[0]],
        [0, cam1_f[1], cam1_c[1]],
        [0, 0, 1]
    ])
    cam2_f = configuration_parameters["cams"][serial_numbers[y]]["intrinsics"]["ir_focal_length"]
    cam2_c = configuration_parameters["cams"][serial_numbers[y]]["intrinsics"]["ir_img_center"]
    cam2_mtx = np.array([
        [cam2_f[0], 0, cam2_c[0]],
        [0, cam2_f[1], cam2_c[1]],
        [0, 0, 1]
    ])


objpoints = []  # Creating vector to store vectors of 3D points for each checkerboard image
imgpoints_1 = []


In [9]:
"""
START RECORDING SOME FRAMES

"""

for device_num in range(len(ctx.devices)):
    pipelines[device_num].start(configs[device_num])

color_images = len(serial_numbers) * [0]
ir_images = len(serial_numbers) * [0]
ir_images_processed = len(serial_numbers) * [0] 
image_count = 0
exposure_d415 = 70000
gain_d415 = 30

set_gain = False

try:
    while True:

        for i in range(len(serial_numbers)):

            sensor = profiles[i].get_device().query_sensors()[0]
            #print(serial_numbers[i])
        
            # if set_gain == False:
            #     try:
            #         sensor.set_option(rs.option.gain, gain_d415)
            #     finally:
            #         set_gain = True
            #sensor.set_option(rs.option.exposure, exposure_d415)

            frames = pipelines[i].wait_for_frames()

            color_frame = frames.get_color_frame()
            if not color_frame:
                continue
            color_images[i] = np.asanyarray(color_frame.get_data())

            ir_frame = frames.first(rs.stream.infrared)
            if not ir_frame:
                continue

            ir_frame_original = np.asanyarray(ir_frame.get_data())
            ir_frame_processed = np.copy(color_images[i])
            corners_1, ids_1, rejectedImgPoints_1 = aruco.detectMarkers(ir_frame_original, ARUCO_DICT, parameters=ARUCO_PARAMETERS)
            #print(len(corners_1))
            if len(corners_1) != 0:
            # Refine detected markers
            # Eliminates markers not part of our board, adds missing markers to the board
                corners_1, ids_1, rejectedImgPoints_1, recoveredIds_1 = aruco.refineDetectedMarkers(
                image=ir_frame_original,
                board=CHARUCO_BOARD,
                detectedCorners=corners_1,
                detectedIds=ids_1,
                rejectedCorners=rejectedImgPoints_1,
                cameraMatrix=cam1_mtx,
                distCoeffs=distCoeffs)
                
            # Only try to find CharucoBoard if we found markers
                if ids_1 is not None and len(ids_1) > 10:
                # Get charuco corners and ids from detected aruco markers
                    response_1, charuco_corners_1, charuco_ids_1 = aruco.interpolateCornersCharuco(
                    markerCorners=corners_1,
                    markerIds=ids_1,
                    image=ir_frame_original,
                    board=CHARUCO_BOARD)
                    

                    if response_1 is not None and response_1 > 20\
                        and len(charuco_corners_1) == len(objp):

                        objpoints.append(objp)
                        imgpoints_1.append(charuco_corners_1)

                        # Outline all of the markers detected in our image
                        ir_frame_processed = aruco.drawDetectedMarkers(ir_frame_processed, corners_1, borderColor=(0, 0, 255))
                        


            ir_images[i] = ir_frame_original
            ir_images_processed[i] = ir_frame_processed

        # Stack all images horizontally
        # images_color = np.hstack(tuple(color_images))
        images_ir = np.hstack(tuple(ir_images_processed))

        # Show images from all cameras
        cv.namedWindow('RealSense', cv.WINDOW_NORMAL)
        cv.imshow('RealSense', images_ir)
        ch = cv.waitKey(1)
        if ch==32:
            image_count +=1
            print("Saving image: ", image_count)
            for i in range(len(serial_numbers)):
                cv.imwrite('./' + serial_numbers[i] + '/calibration_images/image_' + str(image_count) + '.jpg', ir_images[i])

            if image_count == NUM_CALIB_IMGS:
                break

finally:

    # Stop streaming
    for pipeline in pipelines:
        pipeline.stop()

    cv.destroyAllWindows()


: 

: 

## Combining Point Clouds from all Cameras

This final part of the code combines the point clouds obtained from all cameras based on their relative position and orientation.

In [ ]:
"""
GET CALIBRATION DATA
"""

SETTINGS_PATH = './configuration_parameters.json'
param = json.load(open(SETTINGS_PATH))
cams_list = list(param["cams"].keys())
print("cams_list: ", cams_list)
CAM_DATA = [param["cams"][cam] for cam in param["cams"]]  # camera data

In [ ]:
"""
DETERMINING R and T for each cam relative to the first cam
"""


def find_path_to_cam_0(initial_cam):
    unvisited_cams = cams_list.copy()
    min_path = {}
    previous_nodes = {}
    max_value = sys.maxsize

    for cam in unvisited_cams:
        min_path[cam] = max_value

    min_path[cams_list[0]] = 0

    while len(unvisited_cams) > 0:
        current_min_node = None
        for cam in unvisited_cams:
            if current_min_node == None:
                current_min_node = cam
            elif min_path[cam] < min_path[current_min_node]:
                current_min_node = cam

        calibrated_cams = [x for x in list(param["cams"][current_min_node].keys()) if x != "intrinsics"]
        for neighbor in calibrated_cams:
            distance = min_path[current_min_node] + 1
            if distance < min_path[neighbor]:
                min_path[neighbor] = distance
                previous_nodes[neighbor] = current_min_node

        unvisited_cams.remove(current_min_node)

    path = []
    node = initial_cam
    while node != cams_list[0]:
        path.append(node)
        node = previous_nodes[node]

    path.append(cams_list[0])

    return path


In [ ]:
"""
CREATING CAMERA OBJECTS
"""
cam = []
for i in range(len(cams_list)):
    calibrated_cams = [x for x in list(CAM_DATA[i].keys()) if x != "intrinsics"]
    if len(calibrated_cams) == 0:
        print("No stereocalibration data for camera " + cams_list[i])
        continue

    if i == 0:
        rotation = [[1.0, 0.0, 0.0],
                    [0.0, 1.0, 0.0],
                    [0.0, 0.0, 1.0]]
        translation = [0.0, 0.0, 0.0]
        cam.append(Camera.Camera(CAM_DATA[i]["intrinsics"]["img_size"], CAM_DATA[i]["intrinsics"]["ir_focal_length"],
                                 CAM_DATA[i]["intrinsics"]["ir_img_center"], rotation, translation))
        path = find_path_to_cam_0(cams_list[i])


    else:
        path = find_path_to_cam_0(cams_list[i])
        rotation = np.eye(3)
        previous_rotation = np.eye(3)
        translation = np.array([0, 0, 0])
        for j in range(1, len(path)):
            idx = cams_list.index(path[j - 1])
            previous_rotation = CAM_DATA[idx][path[j]]["rotation"]
            translation = np.add(CAM_DATA[idx][path[j]]["translation"], np.matmul(previous_rotation, translation))
            rotation = np.matmul(previous_rotation, rotation)
        print("Current Cam: ", cams_list[i])
        print("path to cam0: ", path)
        print("Final rotation: \n", rotation)
        print("Final translation: ", translation)
        print("___")
        cam.append(Camera.Camera(CAM_DATA[i]["intrinsics"]["img_size"], CAM_DATA[i]["intrinsics"]["ir_focal_length"],
                                 CAM_DATA[i]["intrinsics"]["ir_img_center"], rotation, translation))



In [ ]:


for i in range(len(cams_list)):
    cam[i].add_image(cv.imread("./" + cams_list[i] + "/sample_images/image.jpg"),
                     np.load("./" + cams_list[i] + "/sample_images/depth_map.npy") * 0.001)
    cam[i].point_cloud()
    cam[i].visualize()


In [ ]:

combiner = Camera.Combiner(cam)
combiner.combine()
combiner.visualize()